In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, signal
import warnings
from typing import Dict, Any, Tuple, List
import math

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. SMART TYPE DETECTION
# ==============================================================================
def _is_categorical(s: pl.Series) -> bool:
    """Determines if a series should be treated as categorical."""
    if s.dtype in (pl.String, pl.Categorical, pl.Enum, pl.Boolean):
        return True
    if str(s.dtype) == "Utf8":
        return True
    if s.dtype.is_numeric():
        # FIX: Treat low-cardinality numeric columns (like 0/1 flags, Likert scales) as categorical
        if s.n_unique() <= 15:
            return True
    return False

# ==============================================================================
# 2. SPEED OF LIGHT STATISTICAL ENGINES (Pure NumPy / SciPy FFT)
# ==============================================================================
def _fast_prewhitened_ccf(x: np.ndarray, y: np.ndarray, max_lag: int) -> Tuple[np.ndarray, np.ndarray, int]:
    """Computes Pre-whitened Cross-Correlation using FFT and AR(1) filtering."""
    phi = np.corrcoef(x[:-1], x[1:])[0, 1]
    if np.isnan(phi): phi = 0.0
    x_pw = x[1:] - phi * x[:-1]
    y_pw = y[1:] - phi * y[:-1]
    
    x_pw = (x_pw - np.mean(x_pw)) / (np.std(x_pw) + 1e-12)
    y_pw = (y_pw - np.mean(y_pw)) / (np.std(y_pw) + 1e-12)
    n = len(x_pw)
    
    ccf = signal.correlate(x_pw, y_pw, mode='full', method='fft') / n
    lags = np.arange(-max_lag, max_lag + 1)
    indices = n - 1 + lags
    
    valid_mask = (indices >= 0) & (indices < len(ccf))
    valid_lags = lags[valid_mask]
    valid_ccf = ccf[indices[valid_mask]]
    
    opt_lag = valid_lags[np.argmax(np.abs(valid_ccf))]
    return valid_lags, valid_ccf, int(opt_lag)

def _fast_granger_causality(x: np.ndarray, y: np.ndarray, max_lag: int) -> Dict[str, Any]:
    """Tests Granger Causality using Vector Autoregression (VAR) and F-tests."""
    n = len(x)
    best_f, best_p, best_lag = 0.0, 1.0, 1
    lags_to_test = sorted(list(set([1, 2, 3, 5, max_lag])))
    lags_to_test = [l for l in lags_to_test if l < n // 3]
    
    for lag in lags_to_test:
        Y = y[lag:]
        X_R = np.ones((n - lag, 1 + lag))
        for i in range(1, lag + 1): X_R[:, i] = y[lag - i : n - i]
        
        X_U = np.ones((n - lag, 1 + 2 * lag))
        X_U[:, 1:lag+1] = X_R[:, 1:]
        for i in range(1, lag + 1): X_U[:, lag + i] = x[lag - i : n - i]
        
        beta_R, _, _, _ = np.linalg.lstsq(X_R, Y, rcond=None)
        beta_U, _, _, _ = np.linalg.lstsq(X_U, Y, rcond=None)
        
        rss_R = np.sum((Y - X_R @ beta_R)**2)
        rss_U = np.sum((Y - X_U @ beta_U)**2)
        
        df_num = lag
        df_denom = n - 2 * lag - 1
        if df_denom > 0 and rss_U > 1e-12:
            f_stat = ((rss_R - rss_U) / df_num) / (rss_U / df_denom)
            p_val = stats.f.sf(f_stat, df_num, df_denom)
            if p_val < best_p:
                best_p, best_f, best_lag = p_val, f_stat, lag
                
    return {"f_stat": best_f, "p_value": best_p, "best_lag": best_lag}

def _fast_cointegration(x: np.ndarray, y: np.ndarray) -> Dict[str, Any]:
    """Engle-Granger Cointegration Test via OLS and Fast ADF on residuals."""
    X = np.vstack([x, np.ones(len(x))]).T
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    epsilon = y - X @ beta
    
    diff_eps = np.diff(epsilon)
    eps_lag = epsilon[:-1]
    X_adf = np.vstack([eps_lag, np.ones(len(eps_lag))]).T
    gamma, _, _, _ = np.linalg.lstsq(X_adf, diff_eps, rcond=None)
    resid = diff_eps - X_adf @ gamma 
    
    n_adf = len(diff_eps)
    sigma2 = np.sum(resid**2) / (n_adf - 2)
    var_gamma = sigma2 * np.linalg.inv(X_adf.T @ X_adf)[0, 0]
    t_stat = gamma[0] / np.sqrt(var_gamma + 1e-12)
    
    return {"t_stat": float(t_stat), "is_cointegrated_5pct": bool(t_stat < -3.34), 
            "beta": float(beta[0]), "alpha": float(beta[1])}

def _fast_transfer_entropy(x: np.ndarray, y: np.ndarray, bins: int = 10, lag: int = 1) -> float:
    """Computes Transfer Entropy using vectorized 3D histograms with safe broadcasting."""
    if len(x) <= lag or len(y) <= lag: return 0.0
        
    x_edges = np.percentile(x, np.linspace(0, 100, bins + 1))
    y_edges = np.percentile(y, np.linspace(0, 100, bins + 1))
    
    if x_edges[0] == x_edges[-1]: x_edges = np.linspace(x_edges[0] - 1, x_edges[0] + 1, bins + 1)
    else: x_edges[0], x_edges[-1] = x_edges[0] - 1e-6, x_edges[-1] + 1e-6
        
    if y_edges[0] == y_edges[-1]: y_edges = np.linspace(y_edges[0] - 1, y_edges[0] + 1, bins + 1)
    else: y_edges[0], y_edges[-1] = y_edges[0] - 1e-6, y_edges[-1] + 1e-6
    
    x_bins = np.clip(np.digitize(x, x_edges) - 1, 0, bins - 1)
    y_bins = np.clip(np.digitize(y, y_edges) - 1, 0, bins - 1)
    
    y_future, y_present, x_present = y_bins[lag:], y_bins[:-lag], x_bins[:-lag]
    
    hist_3d, _ = np.histogramdd([y_future, y_present, x_present], bins=bins)
    p_xyz = hist_3d / (hist_3d.sum() + 1e-12)
    
    p_yf_yp, _, _ = np.histogram2d(y_future, y_present, bins=bins)
    p_yf_yp = p_yf_yp / (p_yf_yp.sum() + 1e-12)
    
    p_yp_xp, _, _ = np.histogram2d(y_present, x_present, bins=bins)
    p_yp_xp = p_yp_xp / (p_yp_xp.sum() + 1e-12)
    
    p_yp, _ = np.histogram(y_present, bins=bins)
    p_yp = p_yp / (p_yp.sum() + 1e-12)
    
    p_yf_yp_r = p_yf_yp.reshape(bins, bins, 1)
    p_yp_xp_r = p_yp_xp.reshape(1, bins, bins)
    p_yp_r = p_yp.reshape(1, bins, 1)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        numerator = p_xyz * p_yp_r
        denominator = p_yf_yp_r * p_yp_xp_r
        log_ratio = np.log(numerator / denominator)
        
    valid_mask = (p_xyz > 0) & (denominator > 0)
    te = np.sum(p_xyz[valid_mask] * log_ratio[valid_mask])
    return max(0.0, float(te))

def _fast_lagged_mi(x: np.ndarray, y: np.ndarray, max_lag: int) -> List[Dict[str, Any]]:
    """Computes Lagged Mutual Information for Categorical series."""
    results = []
    bins_x, bins_y = np.max(x) + 2, np.max(y) + 2
    for lag in range(max_lag + 1):
        x_lag, y_lag = (x, y) if lag == 0 else (x[:-lag], y[lag:])
        hist, _, _ = np.histogram2d(x_lag, y_lag, bins=[bins_x, bins_y])
        p_xy = hist / hist.sum()
        p_x, p_y = p_xy.sum(axis=1), p_xy.sum(axis=0)
        mask = p_xy > 0
        mi = np.sum(p_xy[mask] * np.log(p_xy[mask] / (p_x[:, None][mask] * p_y[None, :][mask])))
        results.append({"lag": lag, "mi": float(mi)})
    return results

# ==============================================================================
# 3. MAIN BIVARIATE TIME SERIES DISPATCHER
# ==============================================================================
def bivariate_timeseries_analysis(s1: pl.Series, s2: pl.Series, max_lag: int = 15) -> Dict[str, Any]:
    """
    Brutally optimized Bivariate Time Series Analysis.
    Handles Numeric-Numeric, Numeric-Categorical, and Categorical-Categorical pairs.
    Automatically detects binary/low-cardinality flags (0/1) as Categorical.
    """
    name1 = s1.name if s1.name else "series_1"
    name2 = s2.name if s2.name else "series_2"
    
    df = pl.DataFrame({name1: s1, name2: s2}).with_row_index("time_step")
    df_clean = df.drop_nulls()
    total_rows = len(df_clean)
    
    if total_rows < 20:
        raise ValueError("Dataset too small for valid time series bivariate analysis.")
        
    max_lag = min(max_lag, total_rows // 5)
    
    # SMART TYPE DETECTION (Fixes the 0/1 binary flag issue)
    is_num1 = not _is_categorical(s1)
    is_num2 = not _is_categorical(s2)
    
    step_size = max(1, total_rows // 250_000)
    if step_size > 1:
        indices = pl.int_range(0, total_rows, step=step_size, dtype=pl.UInt32)
        df_plot = df_clean.gather(indices)
    else:
        df_plot = df_clean
        
    time_steps = df_plot.get_column("time_step").to_numpy()
    
    sns.set_theme(style="whitegrid", palette="muted")
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.patch.set_facecolor('#F8F9FA')
    for ax_row in axes:
        for ax in ax_row:
            ax.set_facecolor('#F8F9FA')
            
    results = {"pair": f"{name1} vs {name2}", "total_rows": total_rows}
    
    # ==========================================================================
    # CASE A: NUMERIC vs NUMERIC
    # ==========================================================================
    if is_num1 and is_num2:
        x = df_clean.get_column(name1).to_numpy().astype(float)
        y = df_clean.get_column(name2).to_numpy().astype(float)
        
        lags, ccf_vals, opt_lag = _fast_prewhitened_ccf(x, y, max_lag)
        g_xy = _fast_granger_causality(x, y, max_lag)
        g_yx = _fast_granger_causality(y, x, max_lag)
        coint = _fast_cointegration(x, y)
        te_xy = _fast_transfer_entropy(x, y, lag=abs(opt_lag) if opt_lag != 0 else 1)
        te_yx = _fast_transfer_entropy(y, x, lag=abs(opt_lag) if opt_lag != 0 else 1)
        
        results.update({
            "type": "Numeric-Numeric", "optimal_lag": opt_lag, "max_ccf": float(np.max(np.abs(ccf_vals))),
            "granger_X_to_Y": g_xy, "granger_Y_to_X": g_yx, "cointegration": coint,
            "transfer_entropy_X_to_Y": te_xy, "transfer_entropy_Y_to_X": te_yx
        })
        
        z_x, z_y = (x - np.mean(x))/np.std(x), (y - np.mean(y))/np.std(y)
        axes[0,0].plot(time_steps, z_x[::step_size], color='#0077B6', label=name1, alpha=0.8)
        axes[0,0].plot(time_steps, z_y[::step_size], color='#E63946', label=name2, alpha=0.8)
        axes[0,0].set_title("Standardized Chronological Trend", fontweight='bold', color='#1D3557')
        axes[0,0].legend()
        
        colors = ['#E63946' if l == opt_lag else '#1D3557' for l in lags]
        axes[0,1].bar(lags, ccf_vals, color=colors, width=0.8)
        axes[0,1].axhline(0, color='black', linewidth=1)
        axes[0,1].set_title(f"Pre-Whitened CCF (Optimal Lag: {opt_lag})", fontweight='bold', color='#1D3557')
        
        if opt_lag >= 0: x_lag, y_lag = x[:-opt_lag] if opt_lag > 0 else x, y[opt_lag:] if opt_lag > 0 else y
        else: x_lag, y_lag = x[-opt_lag:], y[:opt_lag]
            
        axes[1,0].scatter(x_lag[::step_size], y_lag[::step_size], color='#0077B6', alpha=0.4, s=15)
        z = np.polyfit(x_lag, y_lag, 1)
        p = np.poly1d(z)
        x_line = np.linspace(np.min(x_lag), np.max(x_lag), 100)
        axes[1,0].plot(x_line, p(x_line), color='#E63946', linewidth=2.5)
        axes[1,0].set_title(f"Scatter at Lag {opt_lag}", fontweight='bold', color='#1D3557')
        
        summary = (f"Granger X→Y: p={g_xy['p_value']:.4f}\nGranger Y→X: p={g_yx['p_value']:.4f}\n"
                   f"Cointegrated (5%): {coint['is_cointegrated_5pct']}\n"
                   f"TE X→Y: {te_xy:.4f} | TE Y→X: {te_yx:.4f}")
        axes[1,1].text(0.5, 0.5, summary, ha='center', va='center', fontsize=14, 
                        family='monospace', bbox=dict(boxstyle='round', facecolor='#A8DADC', alpha=0.5))
        axes[1,1].set_title("Statistical Diagnostics", fontweight='bold', color='#1D3557')
        axes[1,1].axis('off')

    # ==========================================================================
    # CASE B: NUMERIC vs CATEGORICAL (or vice versa)
    # ==========================================================================
    elif is_num1 != is_num2:
        num_col = name1 if is_num1 else name2
        cat_col = name2 if is_num1 else name1
        
        # Force Categorical mapping (Casts 0/1 integers to "0"/"1" strings safely)
        df_clean = df_clean.with_columns(pl.col(cat_col).cast(pl.String).cast(pl.Categorical))
        
        num_vals = df_clean.get_column(num_col).to_numpy().astype(float)
        cat_codes = df_clean.get_column(cat_col).to_physical().to_numpy()
        cat_strings = df_clean.get_column(cat_col).to_numpy() # Keep strings for clean plotting
        
        ss_total = np.var(num_vals) * (len(num_vals) - 1)
        lagged_eta = []
        for lag in range(max_lag + 1):
            c_lag = cat_codes[:-lag] if lag > 0 else cat_codes
            n_lag = num_vals[lag:] if lag > 0 else num_vals
            grand_mean = np.mean(n_lag)
            ss_between = 0.0
            for c in np.unique(c_lag):
                mask = c_lag == c
                if np.sum(mask) > 0:
                    ss_between += np.sum(mask) * (np.mean(n_lag[mask]) - grand_mean)**2
            lagged_eta.append({"lag": lag, "eta_sq": ss_between / ss_total if ss_total > 0 else 0.0})
            
        opt_lag = max(lagged_eta, key=lambda x: x["eta_sq"])["lag"]
        
        c_opt = cat_codes[:-opt_lag] if opt_lag > 0 else cat_codes
        n_opt = num_vals[opt_lag:] if opt_lag > 0 else num_vals
        c_opt_str = cat_strings[:-opt_lag] if opt_lag > 0 else cat_strings # For plotting
        
        groups = [n_opt[c_opt == c] for c in np.unique(c_opt)]
        h_stat, p_kw = stats.kruskal(*groups) if len(groups) > 1 else (0.0, 1.0)
        
        results.update({
            "type": "Numeric-Categorical", "numeric_var": num_col, "categorical_var": cat_col,
            "optimal_lag": opt_lag, "max_eta_squared": max(x["eta_sq"] for x in lagged_eta),
            "kruskal_wallis_p": p_kw, "lagged_eta": lagged_eta
        })
        
        ax1 = axes[0,0]
        ax2 = ax1.twinx()
        ax1.plot(time_steps, num_vals[::step_size], color='#0077B6', label=num_col)
        ax2.scatter(time_steps, cat_codes[::step_size], color='#E63946', alpha=0.5, s=15, label=cat_col)
        ax1.set_title("Chronological Overlay", fontweight='bold', color='#1D3557')
        
        lags = [x["lag"] for x in lagged_eta]
        etas = [x["eta_sq"] for x in lagged_eta]
        axes[0,1].plot(lags, etas, color='#2A9D8F', marker='o', linewidth=2.5)
        axes[0,1].axvline(opt_lag, color='#E63946', linestyle='--')
        axes[0,1].set_title("Lagged Eta-Squared (Predictive Power)", fontweight='bold', color='#1D3557')
        
        # Boxplot using actual string labels so "0" and "1" show up cleanly on the X-axis
        plot_df = pl.DataFrame({"cat": c_opt_str, "num": n_opt})
        top_cats = plot_df.group_by("cat").len().sort("len", descending=True).head(10)["cat"].to_list()
        plot_df = plot_df.filter(pl.col("cat").is_in(top_cats))
        sns.boxplot(x="cat", y="num", data=plot_df.to_pandas(), ax=axes[1,0], palette="mako")
        axes[1,0].set_title(f"Distribution at Lag {opt_lag}", fontweight='bold', color='#1D3557')
        axes[1,0].tick_params(axis='x', rotation=45)
        
        summary = f"Optimal Lag: {opt_lag}\nMax Eta-Sq: {max(etas):.4f}\nKruskal-Wallis p: {p_kw:.4f}"
        axes[1,1].text(0.5, 0.5, summary, ha='center', va='center', fontsize=14, 
                        family='monospace', bbox=dict(boxstyle='round', facecolor='#A8DADC', alpha=0.5))
        axes[1,1].set_title("Statistical Diagnostics", fontweight='bold', color='#1D3557')
        axes[1,1].axis('off')

    # ==========================================================================
    # CASE C: CATEGORICAL vs CATEGORICAL
    # ==========================================================================
    else:
        df_clean = df_clean.with_columns([
            pl.col(name1).cast(pl.String).cast(pl.Categorical),
            pl.col(name2).cast(pl.String).cast(pl.Categorical)
        ])
        
        x_codes = df_clean.get_column(name1).to_physical().to_numpy()
        y_codes = df_clean.get_column(name2).to_physical().to_numpy()
        
        lagged_mi = _fast_lagged_mi(x_codes, y_codes, max_lag)
        opt_lag = max(lagged_mi, key=lambda x: x["mi"])["lag"]
        
        x_lag, y_lag = x_codes[:-1], y_codes[1:]
        hist, _, _ = np.histogram2d(x_lag, y_lag, bins=[np.max(x_codes)+2, np.max(y_codes)+2])
        trans_matrix = hist / (hist.sum(axis=1, keepdims=True) + 1e-12)
        
        results.update({
            "type": "Categorical-Categorical", "optimal_lag": opt_lag,
            "max_mutual_info": max(x["mi"] for x in lagged_mi),
            "lagged_mi": lagged_mi, "transition_matrix_shape": trans_matrix.shape
        })
        
        axes[0,0].scatter(time_steps, x_codes[::step_size], color='#0077B6', alpha=0.6, s=15, label=name1)
        axes[0,0].scatter(time_steps, y_codes[::step_size], color='#E63946', alpha=0.6, s=15, label=name2)
        axes[0,0].set_title("Sequential Category Evolution", fontweight='bold', color='#1D3557')
        axes[0,0].legend()
        
        lags = [x["lag"] for x in lagged_mi]
        mis = [x["mi"] for x in lagged_mi]
        axes[0,1].plot(lags, mis, color='#2A9D8F', marker='o', linewidth=2.5)
        axes[0,1].axvline(opt_lag, color='#E63946', linestyle='--')
        axes[0,1].set_title("Lagged Mutual Information", fontweight='bold', color='#1D3557')
        
        sns.heatmap(trans_matrix, ax=axes[1,0], cmap="mako", cbar=False)
        axes[1,0].set_title("Markov Transition Matrix (Lag 1)", fontweight='bold', color='#1D3557')
        
        summary = f"Optimal Lag: {opt_lag}\nMax MI: {max(mis):.4f}\nMatrix Dim: {trans_matrix.shape}"
        axes[1,1].text(0.5, 0.5, summary, ha='center', va='center', fontsize=14, 
                        family='monospace', bbox=dict(boxstyle='round', facecolor='#A8DADC', alpha=0.5))
        axes[1,1].set_title("Statistical Diagnostics", fontweight='bold', color='#1D3557')
        axes[1,1].axis('off')

    # ==========================================================================
    # 4. GLOBAL AESTHETICS & CONSOLE OUTPUT
    # ==========================================================================
    for ax_row in axes:
        for ax in ax_row:
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#CCCCCC')
            ax.spines['bottom'].set_color('#CCCCCC')
            ax.tick_params(colors='#555555', labelsize=10)
            
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*65}")
    print(f"  BIVARIATE TIME SERIES ANALYSIS: {name1.upper()} vs {name2.upper()}")
    print(f"{'='*65}")
    print(f"  Total Aligned Rows: {total_rows:>15,}")
    print(f"  Analysis Type:      {results['type']:>15}")
    print(f"  Optimal Lag:        {results.get('optimal_lag', 'N/A'):>15}")
    
    if results['type'] == 'Numeric-Numeric':
        print(f"  Max Abs CCF:        {results['max_ccf']:>15.4f}")
        print(f"  Cointegrated (5%):  {str(results['cointegration']['is_cointegrated_5pct']):>15}")
        print(f"  Granger X->Y (p):   {results['granger_X_to_Y']['p_value']:>15.4f}")
        print(f"  Granger Y->X (p):   {results['granger_Y_to_X']['p_value']:>15.4f}")
        print(f"  Transfer Entropy:   {results['transfer_entropy_X_to_Y']:.4f} (X->Y) | {results['transfer_entropy_Y_to_X']:.4f} (Y->X)")
    elif results['type'] == 'Numeric-Categorical':
        print(f"  Max Eta-Squared:    {results['max_eta_squared']:>15.4f}")
        print(f"  Kruskal-Wallis p:   {results['kruskal_wallis_p']:>15.4f}")
    else:
        print(f"  Max Mutual Info:    {results['max_mutual_info']:>15.4f}")
        
    print(f"{'='*65}\n")
    
    return results

### Load Datasets

In [ ]:
north_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/north_raw.parquet")
south_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/south_raw.parquet")

### Daily Max Temp vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["daily_max_temp_celsius"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["daily_max_temp_celsius"], s2=south_df["water_quality_index"], max_lag=7))

### Temp Anomaly vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["temp_anomaly_celsius"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["temp_anomaly_celsius"], s2=south_df["water_quality_index"], max_lag=7))

### Cumulative Heat Index vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["cumulative_heat_index"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["cumulative_heat_index"], s2=south_df["water_quality_index"], max_lag=7))

### Daily Rainfall vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["daily_rainfall_mm"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["daily_rainfall_mm"], s2=south_df["water_quality_index"], max_lag=7))

### Consecutive Dry Days vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["consecutive_dry_days"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["consecutive_dry_days"], s2=south_df["water_quality_index"], max_lag=7))

### Rolling 7D Rainfall vs Water Qualtiy Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["rolling_7d_rainfall_mm"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["rolling_7d_rainfall_mm"], s2=south_df["water_quality_index"], max_lag=7))

### Cumulative Storm Rainfall vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["cumulative_storm_rainfall_mm"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["cumulative_storm_rainfall_mm"], s2=south_df["water_quality_index"], max_lag=7))

### Antecedent Moisture Condition vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["antecedent_moisture_condition"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["antecedent_moisture_condition"], s2=south_df["water_quality_index"], max_lag=7))

### Daily Runoff Volume vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["daily_runoff_volume_m3"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["daily_runoff_volume_m3"], s2=south_df["water_quality_index"], max_lag=7))

### Total Suspended Solids vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["total_suspended_solids_mg_L"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["total_suspended_solids_mg_L"], s2=south_df["water_quality_index"], max_lag=7))

### Nutrient Load Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["nutrient_load_index"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["nutrient_load_index"], s2=south_df["water_quality_index"], max_lag=7))

### Heat X Nutrient Synergy vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["heat_x_nutrient_synergy"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["heat_x_nutrient_synergy"], s2=south_df["water_quality_index"], max_lag=7))

### Cluster Heavy Users Daily Mean Liters vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["cluster_heavy_users_daily_mean_liters"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["cluster_heavy_users_daily_mean_liters"], s2=south_df["water_quality_index"], max_lag=7))

### Cluster Conservationists Daily Mean Liters vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["cluster_conservationists_daily_mean_liters"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["cluster_conservationists_daily_mean_liters"], s2=south_df["water_quality_index"], max_lag=7))

### Cluster Outdoor Landscape Daily Mean Liters vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["cluster_outdoor_landscape_daily_mean_liters"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["cluster_outdoor_landscape_daily_mean_liters"], s2=south_df["water_quality_index"], max_lag=7))

### Cluster Standard Consumers Daily Mean Liters vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["cluster_standard_consumers_daily_mean_liters"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["cluster_standard_consumers_daily_mean_liters"], s2=south_df["water_quality_index"], max_lag=7))

### Drought X Heat Stress vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["drought_x_heat_stress"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["drought_x_heat_stress"], s2=south_df["water_quality_index"], max_lag=7))

### Demand X Runoff Pressure vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["demand_x_runoff_pressure"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["demand_x_runoff_pressure"], s2=south_df["water_quality_index"], max_lag=7))

### Is Weekend vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["is_weekend"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["is_weekend"], s2=south_df["water_quality_index"], max_lag=7))

### Seasons vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["season_label"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["season_label"], s2=south_df["water_quality_index"], max_lag=7))

### Watering Ban Active vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["watering_ban_active"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["watering_ban_active"], s2=south_df["water_quality_index"], max_lag=7))

### Holiday Weekend Flag vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["holiday_weekend_flag"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["holiday_weekend_flag"], s2=south_df["water_quality_index"], max_lag=7))

### Tiered Pricing Regime vs Water Quality Index

In [ ]:
print(bivariate_timeseries_analysis(s1=north_df["tiered_pricing_regime"], s2=north_df["water_quality_index"], max_lag=7))
print(bivariate_timeseries_analysis(s1=south_df["tiered_pricing_regime"], s2=south_df["water_quality_index"], max_lag=7))